In [ ]:
1+1

## document loader

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader,PyPDFLoader
from src.loging.logger import log
from src.exceptions.custom_exceptions import CustomException

logging=log



class DocumentLoader:
    def __init__(self,directory:str,):
        self.directory=directory
        logging.info("initialized docuemnt loader")

    def document_loader(self):
        rew_documents=PyPDFDirectoryLoader(self.directory)
        self.docs=rew_documents.load()
        if self.docs:
            try:
                pdf_files = set()
                for doc in self.docs:
                    source = doc.metadata.get('source', 'unknown')
                    pdf_files.add(source)
                # Log each PDF file
                logging.info(f"✅ Successfully loaded {len(self.docs)}  , documents lent is  {len(pdf_files)} PDF files:")
                for pdf in pdf_files:
                    logging.info(f" 📄 {pdf}")
                
            except Exception as e:
                    logging.error(f"error in loading documents : {e}")
                    raise CustomException(
                        message=f"Failed to load documents {e}",
                        error_detail=e
                    )
        elif not self.docs:
            logging.error(f" No documents found in {self.directory}")
            print(f"⚠️ No documents found in {self.directory}")
         
        return self.docs
    





In [ ]:
data = DocumentLoader("../data/")

In [ ]:
docs=data.document_loader()

In [ ]:
docs[0].metadata

In [ ]:
docs[0].metadata.get("source")

## embedings

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from src.loging.logger import log
from src.exceptions.custom_exceptions import CustomException
import sys

logging=log


class Embeddings:

    def __init__(self,
        model_name:str="sentence-transformers/all-MiniLM-L6-v2",
        device: str = "cpu",
        normalize_embeddings: bool = True,
        batch_size: int = 32):

        self.model_name = model_name
        self.device = device
        self.normalize_embeddings = normalize_embeddings
        self.batch_size = batch_size
        self._embeddings = None
        logging.info("embedding get initialized")


    def initializing_embedding(self):
        if self._embeddings is None:
            try:
                self.embeddings = HuggingFaceEmbeddings(
                        model_name=self.model_name,
                        model_kwargs={'device': self.device},
                        encode_kwargs={
                            'normalize_embeddings': self.normalize_embeddings,
                            'batch_size': self.batch_size
                        }
                    )
                log.info(f"initialiased embeding model {HuggingFaceEmbeddings.__class__.__name__} with model name : {self.model_name}")
            except Exception as e:
                    logging.error(f"error during initilizing embedings : {e}")
                    raise CustomException(
                        f"Failed to initialize embeddings with model {self.model_name} or their is an error in embeding models {e}",
                        sys
                    )
        return self.embeddings

## chunker

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker   # ← FIXED
from langchain_community.document_loaders import PyPDFDirectoryLoader
from src.loging.logger import log
from src.exceptions.custom_exceptions import CustomException

logging=log



class TextSpliter:
    def __init__(self,embeding_model,persist_directory:str="vectorestore_VDB"):
        self.embeding=embeding_model
        self.persist_directory=persist_directory
        


    
    def split_documents(self,documents):
        # Split
        try:
            text_splitter = SemanticChunker(
                            embeddings=self.embeding,
                            breakpoint_threshold_type="percentile"
                        )

           
            chunks = text_splitter.split_documents(documents)
            logging.info(f"✅ Split {len(documents)} documents into {len(chunks)} chunks")
            return chunks
        except Exception as e:
            logging.error(f"❌ Error splitting documents: {e}")
            raise CustomException(f"Failed to chunk  {e}",sys)






## retriver

In [ ]:
from langchain_chroma import Chroma
from src.loging.logger import log
from src.exceptions.custom_exceptions import CustomException

logging=log
from pathlib import Path

class VectorStore:
    def __init__(self, embeddings, persist_directory):
        self.embeddings = embeddings
        self.persist_directory = persist_directory
        self.vectorstore = None
        Path(persist_directory).mkdir(parents=True, exist_ok=True)
        logging.info(f"✅ VectorStore initialized with persist_dir: {persist_directory}")

    def create_from_documents(self, documents):
        """Create vectorstore from documents"""
        try:
            self.vectorstore = Chroma.from_documents(
                documents=documents,
                embedding=self.embeddings,
                persist_directory=self.persist_directory
            )
            logging.info(f"✅ Created vectorstore with {len(documents)} documents")
            return self.vectorstore
        except Exception as e:
            logging.error(f"❌ Error creating vectorstore: {e}")
            raise CustomException("Failed to create vectorstore", e)

    def load_existing(self):
        """Load existing vectorstore"""
        try:
            self.vectorstore = Chroma(
                embedding_function=self.embeddings,
                persist_directory=self.persist_directory
            )
            logging.info(f"✅ Loaded existing vectorstore from {self.persist_directory}")
            print(f"✅ Loaded existing vectorstore from {self.persist_directory}")
            return self.vectorstore
        except Exception as e:
            logging.error(f"❌ Error loading vectorstore: {e}")
            raise CustomException("Failed to load vectorstore", e)

    def get_retriever(self, k: int = 4):
        """Get retriever from vectorstore"""
        if self.vectorstore is None:
            raise CustomException("Vectorstore not created yet. Call create_from_documents first.")
        return self.vectorstore.as_retriever(search_type="mmr",
                                             search_kwargs={"k": k,
                                                            "fetch_k": 20,
                                                            "lambda_mult": 0.5 })

In [ ]:
data = DocumentLoader("../data/PLAN_COMPTABLE")
docs=data.document_loader()
embeding_model=Embeddings()
embeding=embeding_model.initializing_embedding()
text_splietr=TextSpliter(embeding)
chunks=text_splietr.split_documents(docs)
vectore_store=VectorStore(embeding,"artifacts/vectorestore/CGI")
vectore_store.create_from_documents(chunks)
store=vectore_store.get_retriever()

In [ ]:
print(chunks[2].page_content)

In [ ]:
store.invoke("tva recuperable sur charge")

In [1]:
from src.PipeLine.pipeline import  RagPipeLine

c:\dev\aaa\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pipeline=RagPipeLine(data_dir="../data/PLAN_COMPTABLE",persist_dir="artifacts/vectorestore/plan_comptable",force_rebuild=False)


🚀 RAG Pipeline initialized
📂 Data: ../data/PLAN_COMPTABLE
💾 Persist: artifacts/vectorestore/plan_comptable
🔍 Vectorstore exists: True



In [3]:
pip=pipeline.run()

2026-05-21 20:30:59,916 - INFO - embedding get initialized
2026-05-21 20:31:00,254 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 20:31:00,256 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-21 20:31:00,304 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-05-21 20:31:00,468 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 20:31:00,513 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_

🔄 Loading existing vectorstore...
✅ Loaded existing vectorstore from artifacts/vectorestore/plan_comptable


2026-05-21 20:31:04,728 - INFO - ✅ Loaded existing vectorstore from artifacts/vectorestore/plan_comptable


✅ Loaded existing vectorstore from artifacts/vectorestore/plan_comptable


In [4]:
result=pip.invoke("les cadaux publicitaire dans  IR")
for i in range(len(result)):
    print(result[i].page_content)
    print("___________________________________________")

4428 Autres clients créditeurs 
443 Personnel - créditeur 
4432 Rémunérations dues au personnel 
4433 Dépôts du personnel créditeurs 
4434 Oppositions sur salaires 
4437 Charges du personnel à payer 
4438 Personnel - autres créditeurs 
444 Organismes sociaux 
4441 Caisse Nationale de la Sécurité Sociale 
4443 Caisses de retraite 
4445 Mutuelles 
4447 Charges sociales à payer 
4448 Autres organismes sociaux 
445 Etat - créditeur 
4452 Etat Impôts, taxes et assimilés 
44521 Etat, taxe urbaine et taxe d'édilité 
44522 Etat, patente 
44525 Etat, IGR 
4453 Etat, impôts sur les résultats 
4455 Etat, TVA facturée 
4456 Etat, TVA due (suivant déclarations) 
4457 Etat, impôts et taxes à payer 
4458 Etat - autres comptes créditeurs 
446 Comptes d'associés - créditeurs 
4461 Associés - capital à rembourser 
4462 Associés - versements reçus sur augmentation de capital 
4463 Comptes courants des associés créditeurs 
4464 Associés - opérations faites en commun 
4465 Associés - dividendes à payer 
44

In [5]:
result

[Document(id='8f512bb3-98bc-4954-8809-0ec65e7e5c05', metadata={'moddate': '2021-02-16T14:31:31+00:00', 'creator': 'Microsoft® Word 2016', 'title': 'Plan cptable Maroc.PDF', 'total_pages': 24, 'source': '..\\data\\PLAN_COMPTABLE\\Plan_Comptable_marocain.pdf', 'page': 8, 'creationdate': '2021-02-16T14:31:27+00:00', 'producer': 'www.ilovepdf.com', 'author': 'Pierre', 'page_label': '9'}, page_content="4428 Autres clients créditeurs \n443 Personnel - créditeur \n4432 Rémunérations dues au personnel \n4433 Dépôts du personnel créditeurs \n4434 Oppositions sur salaires \n4437 Charges du personnel à payer \n4438 Personnel - autres créditeurs \n444 Organismes sociaux \n4441 Caisse Nationale de la Sécurité Sociale \n4443 Caisses de retraite \n4445 Mutuelles \n4447 Charges sociales à payer \n4448 Autres organismes sociaux \n445 Etat - créditeur \n4452 Etat Impôts, taxes et assimilés \n44521 Etat, taxe urbaine et taxe d'édilité \n44522 Etat, patente \n44525 Etat, IGR \n4453 Etat, impôts sur les ré